## **Environment Setup**

In [2]:
# Standard library imports
import sys
import os

# Third-party imports
import numpy as np

# Configure path to custom modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

print("✅ Environment setup complete")
print(f"📂 Working directory: {os.getcwd()}")

✅ Environment setup complete
📂 Working directory: /Users/LucaZagaia/Public/Documents/University/TUB/Bachelorthesis/5_Predictive-Maintenance-Repo/Predictive-Maintenance/detection/notebooks_voraus_AD_dataset


## **Dataset Review (voraus AD)**

### 📊 **voraus Anomaly Detection Dataset**

---

#### **Quick Overview**

| Aspect | Details |
|--------|---------|
| **Task** | Pick-and-place with 6-axis Yu-Cobot robot |
| **Format** | Apache Parquet (11.17 GB raw, using 100 Hz version) |
| **Samples** | 2,122 time series (948 train, 1,174 test) |
| **Anomalies** | 12 types (755 anomalous samples in test) |
| **Signals** | 130 channels (78 mechanical + 52 electrical) |
| **Frequency** | 100 Hz (downsampled), ~11s per sample (~1,100 timesteps) |
| **Task Type** | Semi-supervised anomaly detection |

#### **Application Setup**

**Pick-and-Place Cycle:**
1. Camera scans → detects random can position (241g nominal)
2. Robot grips can with vacuum gripper
3. Transports to fixed target position
4. Places and returns to initial position

**Key Feature:** Continuous variance via random placement (not discrete positions like AURSAD)

#### **Sensor Signals (130 Total)**

**Per-Axis Signals (21 × 6 axes = 126):**
- **Mechanical (13):** Joint/motor positions & velocities, 2× torque sensors, inertia, power
- **Electrical (8):** Motor currents (Iq, Id), voltages, power

**Global Signals (9):** Robot/system voltage, current, IO current

**Metadata (excluded from ML):** Time, sample ID, anomaly flag, category, variant, active status

#### **12 Anomaly Types**

| Category | Type | Samples | Description |
|----------|------|---------|-------------|
| **Axis Wear** | Additional friction | 144 | Increased friction (2 levels × 6 axes) |
| | Miscommutation | 89 | Motor calibration errors (6 axes) |
| **Gripping** | Misgrip of can | 11 | Gripper fails to grip |
| | Losing the can | 74 | Can detaches during movement |
| **Process** | Additional axis weight | 156 | Unremoved components (3 weights × 6 axes) |
| | Collision with foam | 72 | Lightweight object impacts (2 sizes) |
| | Collision with cables | 48 | Hanging cable interactions (2 types) |
| | Collision with cardboard | 22 | Box collisions (random positions) |
| | Varying can weight | 80 | Wrong weights: 41g-448g vs 241g nominal |
| | Cable routed at robot | 10 | Tool cables causing friction |
| | Invalid gripping position | 12 | Conveyor timing errors |
| | Unstable platform | 37 | Mobile platform vibrations (3 levels) |

**Design Principles:**
- ✅ Subtle anomalies (don't trigger safety systems)
- ✅ 7/12 types generalizable to other robot applications
- ✅ Multiple intensity levels per type
- ✅ Covers robot, gripper, environment, process errors

#### **Dataset Partitioning**

**Training:** 948 samples (normal only) - recorded chronologically first  
**Test:** 419 normal + 755 anomalous - recorded alternately  
**Rationale:** Simulates real deployment where training precedes anomaly emergence

#### **Recommended Models**

| Method | AUROC | Notes |
|--------|-------|-------|
| **MVT-Flow** (Normalizing Flows) | 93.6% | Best overall, temporal localization |
| Hidden Markov Models | 87.4% | Good for collisions |
| LSTM-VAE | 86.7% | Handles time-series well |
| Conv Autoencoder | 85.2% | Moderate performance |
| ❌ PCA | 80.0% | Too simple for temporal patterns |
| ❌ 1-NN | 77.5% | Doesn't capture dynamics |

**Key Insights:**
- Axis wear easiest to detect (99.8% for miscommutation)
- Collisions more challenging (84-88% AUROC)
- 100 Hz sampling sufficient for most anomaly types
- Requires time-series models (not single-snapshot)

#### **Comparison to Other Datasets**

| | voraus-AD | AURSAD | C-MAPSS | UCI |
|---|-----------|---------|---------|-----|
| **Task** | Pick-and-place | Screwdriver | Turbofan RUL | Tool wear |
| **Data type** | Time-series | Time-series | Degradation cycles | Single snapshots |
| **Frequency** | 100 Hz (used) | 100 Hz | Per cycle | N/A |
| **Signals** | 130 | 125 | 21 | 6 |
| **Anomalies** | 12 types | 4 types | N/A | 5 failure modes |
| **Train samples** | 948 | 1,420 | 100-260 | 8,000 |
| **Target** | Binary/multi-class | Binary/multi-class | RUL (continuous) | Multi-class |
| **Model type** | Time-series ML | Time-series ML | LSTM/CNN | Random Forest |

#### **Use Cases**

**✅ Suitable for:**
- Semi-supervised anomaly detection (no training anomalies)
- Real-time robot health monitoring
- Temporal anomaly localization
- Collision and wear detection

**❌ Not suitable for:**
- RUL prediction (no degradation labels)
- Supervised learning with balanced anomaly training
- Single-snapshot/static models

#### **License & Citation**

📄 **License:** CC BY-NC-SA 4.0  
🔗 **Dataset:** https://www.tnt.uni-hannover.de/vorausAD  
📚 **Paper:** Brockmann et al. (2023), arXiv:2311.04765

---

**Bottom Line:** Multivariate time-series dataset (100 Hz) requiring deep learning for semi-supervised anomaly detection in collaborative robotics.

###